In [2]:
# ============================================================
# 任务：
# 基于给定股票列表，抓取并整理如下字段：
#   code, name, year, 证监会行业分类, 上市时间, 注册地址
#
# 时间范围：
#   2010-2025
#
# 缺失值处理：
#   若某年“证监会行业分类”缺失，则用相邻年份的最近值替代
#
# 依赖安装：
#   pip install akshare pandas tqdm openpyxl
#
# 输出文件：
#   stock_csrc_industry_panel_2010_2025.csv
#   stock_csrc_industry_panel_2010_2025.xlsx
# ============================================================

from pathlib import Path
import re
import time
import pandas as pd
import akshare as ak
from tqdm import tqdm


# ============================================================
# 一、基础参数
# ============================================================

# 你给定的股票列表
stocks = {
    "000001": "平安银行",
    "000002": "万科A",
    "600519": "贵州茅台",
    "300750": "宁德时代",
    "601318": "中国平安",
    "002594": "比亚迪",
    "600036": "招商银行",
    "601012": "隆基绿能",
    "000333": "美的集团",
    "600276": "恒瑞医药",
}

# 时间范围
START_YEAR = 2010
END_YEAR   = 2025
YEARS = list(range(START_YEAR, END_YEAR + 1))

# 输出文件
OUT_CSV  = Path("stock_csrc_industry_panel_2010_2025.csv")
OUT_XLSX = Path("stock_csrc_industry_panel_2010_2025.xlsx")

# 每次请求后稍作停顿，避免请求过快
SLEEP_SECONDS = 0.3


# ============================================================
# 二、辅助函数
# ============================================================

def clean_text(x):
    """
    将输入统一清洗为字符串。
    None、NaN、空白字符等统一处理为空字符串。
    """
    if x is None:
        return ""
    if pd.isna(x):
        return ""
    s = str(x).strip()
    if s.lower() in {"nan", "none", "nat"}:
        return ""
    return s


def normalize_code(code):
    """
    将股票代码规范化为 6 位字符串。
    例如：
    1 -> 000001
    '2594' -> 002594
    """
    s = clean_text(code)
    s = re.sub(r"\D", "", s)
    return s.zfill(6)


def format_date_yyyymmdd(x):
    """
    将日期统一转为 YYYYMMDD 格式字符串。
    若无法识别，则返回空字符串。
    """
    s = clean_text(x)
    if not s:
        return ""

    dt = pd.to_datetime(s, errors="coerce")
    if pd.isna(dt):
        return ""

    return dt.strftime("%Y%m%d")


def build_csrc_industry_string(df_sub):
    """
    将某个时点有效的证监会行业分类记录拼接成一个字符串。

    拼接顺序：
    行业门类 -> 行业大类 -> 行业中类 -> 行业次类

    例如可能得到：
    制造业-电气机械和器材制造业-电池制造
    """
    if df_sub is None or df_sub.empty:
        return ""

    latest_date = df_sub["变更日期"].max()
    temp = df_sub[df_sub["变更日期"] == latest_date].copy()

    values = []

    for col in ["行业门类", "行业大类", "行业中类", "行业次类"]:
        if col in temp.columns:
            vals = (
                temp[col]
                .dropna()
                .astype(str)
                .str.strip()
                .replace("", pd.NA)
                .dropna()
                .unique()
                .tolist()
            )
            for v in vals:
                if v not in values:
                    values.append(v)

    return "-".join(values)


def get_profile_info(code):
    """
    抓取公司概况信息。
    返回：
    - 上市时间
    - 注册地址

    若接口失败，则返回空值。
    """
    try:
        df = ak.stock_profile_cninfo(symbol=code)

        if df is None or df.empty:
            return {
                "上市时间": "",
                "注册地址": ""
            }

        row = df.iloc[0]

        list_date = format_date_yyyymmdd(row.get("上市日期", ""))
        register_addr = clean_text(row.get("注册地址", ""))

        return {
            "上市时间": list_date,
            "注册地址": register_addr
        }

    except Exception as e:
        print(f"[公司概况接口失败] code={code}, error={e}")
        return {
            "上市时间": "",
            "注册地址": ""
        }


def get_industry_change_info(code):
    """
    抓取某只股票全部行业变动记录，并只保留“证监会”分类标准。

    返回一个 DataFrame。
    若失败，则返回空 DataFrame。
    """
    try:
        df = ak.stock_industry_change_cninfo(
            symbol=code,
            start_date="19900101",
            end_date=pd.Timestamp.today().strftime("%Y%m%d")
        )

        if df is None or df.empty:
            return pd.DataFrame()

        df = df.copy()
        df["变更日期"] = pd.to_datetime(df["变更日期"], errors="coerce")

        # 只保留证监会行业分类
        if "分类标准" in df.columns:
            df = df[df["分类标准"].astype(str).str.contains("证监会", na=False)].copy()

        # 排序，便于后续按年份取年末有效值
        df = df.sort_values("变更日期").reset_index(drop=True)

        return df

    except Exception as e:
        print(f"[行业变动接口失败] code={code}, error={e}")
        return pd.DataFrame()


def get_industry_for_year(industry_df, year):
    """
    对于某只股票，返回 year 年年末有效的证监会行业分类。

    逻辑：
    1. 找到所有“变更日期 <= 当年12月31日”的记录
    2. 取其中最新日期对应的记录
    3. 拼接成行业字符串

    若当年之前没有任何记录，则返回空字符串。
    """
    if industry_df is None or industry_df.empty:
        return ""

    year_end = pd.Timestamp(f"{int(year)}-12-31")
    temp = industry_df[industry_df["变更日期"] <= year_end].copy()

    if temp.empty:
        return ""

    return build_csrc_industry_string(temp)


def fill_nearest_values(series):
    """
    使用“最近相邻年份”的值填补缺失值。

    规则：
    1. 对于缺失年份，寻找前后最近的非缺失年份
    2. 哪边距离更近，就使用哪边的值
    3. 若距离相同，优先使用前一年的值
    4. 若只有一侧有值，则使用该侧
    5. 若整列都缺失，则保持缺失

    输入：
    series 的索引必须已经按年份升序排列
    """
    s = series.copy()
    s = s.replace("", pd.NA)

    non_missing_positions = [i for i, v in enumerate(s.tolist()) if pd.notna(v)]

    # 若全部缺失，直接返回
    if len(non_missing_positions) == 0:
        return s.fillna("")

    values = s.tolist()

    for i, v in enumerate(values):
        if pd.notna(v):
            continue

        # 找左侧最近非缺失位置
        left_candidates = [p for p in non_missing_positions if p < i]
        left_pos = max(left_candidates) if left_candidates else None

        # 找右侧最近非缺失位置
        right_candidates = [p for p in non_missing_positions if p > i]
        right_pos = min(right_candidates) if right_candidates else None

        if left_pos is None and right_pos is None:
            values[i] = ""
        elif left_pos is None:
            values[i] = values[right_pos]
        elif right_pos is None:
            values[i] = values[left_pos]
        else:
            left_dist = i - left_pos
            right_dist = right_pos - i

            # 距离相同，优先用前一年的值
            if left_dist <= right_dist:
                values[i] = values[left_pos]
            else:
                values[i] = values[right_pos]

    return pd.Series(values, index=s.index)


# ============================================================
# 三、抓取静态信息与行业变动信息
# ============================================================

profile_dict = {}
industry_dict = {}

print("开始抓取公司概况与行业变动信息...")

for code, name in tqdm(stocks.items()):
    code = normalize_code(code)

    # 抓取公司概况
    profile_dict[code] = get_profile_info(code)
    time.sleep(SLEEP_SECONDS)

    # 抓取行业变动
    industry_dict[code] = get_industry_change_info(code)
    time.sleep(SLEEP_SECONDS)


# ============================================================
# 四、按 2010-2025 生成面板数据
# ============================================================

rows = []

print("开始生成年度面板数据...")

for code, name in tqdm(stocks.items()):
    code = normalize_code(code)

    profile_info = profile_dict.get(
        code,
        {"上市时间": "", "注册地址": ""}
    )
    industry_df = industry_dict.get(code, pd.DataFrame())

    for year in YEARS:
        industry_name = get_industry_for_year(industry_df, year)

        rows.append({
            "code": code,
            "name": name,
            "year": year,
            "证监会行业分类": industry_name,
            "上市时间": profile_info.get("上市时间", ""),
            "注册地址": profile_info.get("注册地址", "")
        })

panel_df = pd.DataFrame(rows)


# ============================================================
# 五、对“证监会行业分类”按最近相邻年份补缺
# ============================================================

# 先按股票代码、年份排序
panel_df = panel_df.sort_values(["code", "year"]).reset_index(drop=True)

# 对每只股票分别补缺
panel_df["证监会行业分类"] = (
    panel_df.groupby("code", group_keys=False)["证监会行业分类"]
    .apply(fill_nearest_values)
)

# 再把剩余的缺失统一处理为空字符串
panel_df["证监会行业分类"] = panel_df["证监会行业分类"].fillna("")


# ============================================================
# 六、保存结果
# ============================================================

panel_df.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
panel_df.to_excel(OUT_XLSX, index=False)

print("\n处理完成")
print(f"CSV 文件已保存：{OUT_CSV.resolve()}")
print(f"Excel 文件已保存：{OUT_XLSX.resolve()}")

print("\n结果预览：")
print(panel_df.head(20))

开始抓取公司概况与行业变动信息...


100%|██████████| 10/10 [00:31<00:00,  3.16s/it]


开始生成年度面板数据...


100%|██████████| 10/10 [00:00<00:00, 14.94it/s]



处理完成
CSV 文件已保存：D:\github_lianxh\dsfin\Lecture\data_manage\stock_csrc_industry_panel_2010_2025.csv
Excel 文件已保存：D:\github_lianxh\dsfin\Lecture\data_manage\stock_csrc_industry_panel_2010_2025.xlsx

结果预览：
      code  name  year     证监会行业分类      上市时间                    注册地址
0   000001  平安银行  2010  金融业-货币金融服务  19910403      广东省深圳市罗湖区深南东路5047号
1   000001  平安银行  2011  金融业-货币金融服务  19910403      广东省深圳市罗湖区深南东路5047号
2   000001  平安银行  2012  金融业-货币金融服务  19910403      广东省深圳市罗湖区深南东路5047号
3   000001  平安银行  2013  金融业-货币金融服务  19910403      广东省深圳市罗湖区深南东路5047号
4   000001  平安银行  2014  金融业-货币金融服务  19910403      广东省深圳市罗湖区深南东路5047号
5   000001  平安银行  2015  金融业-货币金融服务  19910403      广东省深圳市罗湖区深南东路5047号
6   000001  平安银行  2016  金融业-货币金融服务  19910403      广东省深圳市罗湖区深南东路5047号
7   000001  平安银行  2017  金融业-货币金融服务  19910403      广东省深圳市罗湖区深南东路5047号
8   000001  平安银行  2018  金融业-货币金融服务  19910403      广东省深圳市罗湖区深南东路5047号
9   000001  平安银行  2019  金融业-货币金融服务  19910403      广东省深圳市罗湖区深南东路5047号
10  000001  平安银行  2020  金融业-货币金融服务  199